In [41]:
%pip install -q datasets pandas scikit-learn joblib python-dotenv langchain-core langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [42]:
import os
import pandas as pd
import joblib

from datasets import load_dataset
from dotenv import load_dotenv

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [43]:
dataset = load_dataset("batuhanmtl/job-skill-set")

df = dataset["train"].to_pandas()


In [44]:
dataset

DatasetDict({
    train: Dataset({
        features: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'],
        num_rows: 1167
    })
})

In [45]:
df.sample(5)

,job_id,category,job_title,job_description,job_skill_set
466,3906077073,SALES,Counter Sales,JOB DESCRIPTIONCOUNTER SALESUNITED REFRIGERATI...,"['Customer service', 'Computer skills', 'Commu..."
961,3905825078,BUSINESS-DEVELOPMENT,Business Development Officer NY/NJ,"MAKING TRADE HAPPEN\nCOFACE IS A TEAM OF 4,500...","['B2B Sales', 'Client Acquisition', 'Consultat..."
1073,3904422441,BUSINESS-DEVELOPMENT,"Associate, Corporate Business Development","COMPANY DESCRIPTION\n\nAT SERVICENOW, OUR TECH...","['market research', 'analytical skills', 'nego..."
1033,3906227812,BUSINESS-DEVELOPMENT,Business Development Representative - First Ai...,REQUISITION NUMBER: 169701\n\nJOB DESCRIPTION\...,"['sales experience', 'data analysis', 'prospec..."
400,3902934493,INFORMATION-TECHNOLOGY,Information Technology Specialist 2 (NY HELPs)...,MINIMUM QUALIFICATIONS\n\nOPEN COMPETITIVE: A ...,"['computer science', 'MS Office (Word, PowerPo..."


In [ ]:
required_columns = ["job_id","category","job_title","job_description","job_skill_set"
]

df = df[required_columns].fillna("").copy()

print(df.shape)
df.head()

(1167, 5)


,job_id,category,job_title,job_description,job_skill_set
0,3902668440,HR,Sr Human Resource Generalist,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,"['employee relations', 'talent acquisition', '..."
1,3905823748,HR,Human Resources Manager,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,"['Talent Acquisition', 'Employee Performance M..."
2,3905854799,HR,Director of Human Resources,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,"['Human Resources Management', 'Recruitment', ..."
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,"['talent management', 'organizational developm..."
4,3906250451,HR,Human Resources Generalist (Hybrid Role),DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,"['Microsoft Office', 'Data analysis', 'Employe..."


In [47]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():

    doc = Document(
        page_content=(
            f"Job Title: {row['job_title']}\n"
            f"Category: {row['category']}\n"
            f"Skills: {row['job_skill_set']}\n"
            f"Job Description: {row['job_description']}"
        ),

        metadata={
            "job_id": str(row["job_id"]),
            "category": str(row["category"]),
            "job_title": str(row["job_title"]),
            "job_skill_set": str(row["job_skill_set"])
        }
    )

    documents.append(doc)

print("Total documents:", len(documents))
print(documents[0])

Total documents: 1167
page_content='Job Title: Sr Human Resource Generalist
Category: HR
Skills: ['employee relations', 'talent acquisition', 'performance management', 'compensation and benefits', 'employment laws and regulations', 'analytical skills', 'employee benefits administration', 'HR policies and procedures', 'recruitment strategies', 'FMLA', 'ADA', 'communication', 'interpersonal skills', 'collaboration', 'influence', 'change management', 'professional development', 'payroll processing', 'job design', 'salary administration', 'human capital management system', 'HR audits and assessments', 'FLSA', 'ERISA', 'EEOC guidelines', 'confidentiality', 'intellectual curiosity']
Job Description: SUMMARY
THE SR. HR GENERALIST PROVIDES HR EXPERTISE ACROSS VARIOUS FUNCTIONS INCLUDING EMPLOYEE RELATIONS, TALENT ACQUISITION, PERFORMANCE MANAGEMENT, PAYROLL, COMPENSATION AND BENEFITS. THE SR. HR GENERALIST HAS RESPONSIBILITIES THAT ARE STRATEGIC, ADMINISTRATIVE, AND CONSULTATIVE. THIS ROLE PRO

In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 14860
page_content='Job Title: Sr Human Resource Generalist
Category: HR' metadata={'job_id': '3902668440', 'category': 'HR', 'job_title': 'Sr Human Resource Generalist', 'job_skill_set': "['employee relations', 'talent acquisition', 'performance management', 'compensation and benefits', 'employment laws and regulations', 'analytical skills', 'employee benefits administration', 'HR policies and procedures', 'recruitment strategies', 'FMLA', 'ADA', 'communication', 'interpersonal skills', 'collaboration', 'influence', 'change management', 'professional development', 'payroll processing', 'job design', 'salary administration', 'human capital management system', 'HR audits and assessments', 'FLSA', 'ERISA', 'EEOC guidelines', 'confidentiality', 'intellectual curiosity']"}


In [49]:
print(chunks[42].page_content)

Job Description: JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO)INDUSTRY: ENTERTAINMENT & MEDIA 
JOB DESCRIPTION:EAST 57TH STREET PARTNERS IS SEEKING AN EXPERIENCED AND INNOVATIVE CHIEF HUMAN RESOURCES OFFICER (CHRO) FOR A LEADING ENTERTAINMENT AND MEDIA COMPANY. THIS ROLE IS PERFECT FOR A STRATEGIC HR LEADER WHO HAS A PASSION FOR THE DYNAMIC WORLD OF ENTERTAINMENT AND MEDIA, AND IS SKILLED IN MANAGING DIVERSE TALENT IN A CREATIVE ENVIRONMENT.


In [50]:
texts = [chunk.page_content for chunk in chunks]

vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)

tfidf_matrix = vectorizer.fit_transform(texts)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (14860, 20000)


In [51]:
joblib.dump(
    {
        "vectorizer": vectorizer,
        "matrix": tfidf_matrix,
        "chunks": chunks
    },"tfidf_index.joblib"
)

print("Index saved successfully.")


Index saved successfully.


In [52]:
def retrieve(question):
    scores = cosine_similarity(vectorizer.transform([question]),tfidf_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:4]

    return "\n\n".join(chunks[i].page_content
        for i in top_indices if scores[i] > 0
    )

In [53]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=api_key,
    temperature=0
)

In [55]:
prompt = ChatPromptTemplate([
  ('system' , 'You are a helpful job information expert.'),
   ('human' , 'Context:\n {context}\n\n Question:\n{question}')

])

In [56]:
rag_chain = (
    {
        "context": RunnableLambda(retrieve),
        "question": RunnablePassthrough()
    }| prompt| model| StrOutputParser()
)

In [57]:
question = "What skills are required for a data scientist?"

answer = rag_chain.invoke(question)

print(answer)

Based on the provided job description context, the key skills and capabilities required for this Data Scientist role include:

* **Analytical Problem-Solving:** Ability to solve key strategic issues using structured analytical approaches.
* **Advanced Analytics & Big Data:** Expertise in applying advanced analytics and utilizing a big data-driven approach to address business issues.
* **Data Communication & Stakeholder Management:** Skill in bringing the power of data to key stakeholders and translating complex data into actionable insights.
* **Strategic Collaboration:** Working hand-in-hand with Finance, Strategy, and Leadership teams to guide business decisions.


In [58]:
joblib.dump(
    {
        "vectorizer": vectorizer,
        "matrix": tfidf_matrix,
        "chunks": chunks
    },"tfidf_index.joblib"
)

print("Index saved successfully!")

Index saved successfully!


In [59]:
index = joblib.load("tfidf_index.joblib")

print("Total chunks:", len(index["chunks"]))
print("Matrix shape:", index["matrix"].shape)

Total chunks: 14860
Matrix shape: (14860, 20000)
